# 🏢 Analyse Stratégique — Compagnie d'Assurance Africaine
## Juillet 2020 – Décembre 2025

---

**Auteur :** DAVID SOUWAN · Data Analyst  
**Périmètre :** 5 pays · 4 produits · 300 agents · 1 200 contrats  
**Objectif :** Transformer les données en décisions stratégiques actionnables

---

### Problématiques business
- Rentabilité instable malgré une croissance visible du portefeuille
- Taux de sinistres préoccupant sans visibilité sur les causes
- Performances géographiques très disparates
- Efficacité commerciale des agents non mesurée

### Plan d'analyse
1. Audit & chargement des données
2. Nettoyage et préparation
3. Construction du dataset maître (df_master)
4. KPIs globaux
5. Analyse par produit
6. Analyse géographique
7. Analyse temporelle
8. Gestion du risque
9. Analyse clients & CLV
10. Performance des agents
11. Visualisations stratégiques
12. Insights & Recommandations

---
## 0️⃣ Configuration & Imports

In [3]:
# ── Imports principaux ────────────────────────────────────────────────────────
import pandas as pd           # Manipulation et analyse des données tabulaires.
import numpy as np            # Calculs numériques et gestion des valeurs nulles.
import matplotlib.pyplot as plt  # Création de graphiques statiques professionnels.
import matplotlib.patches as mpatches  # Éléments visuels personnalisés pour les légendes.
import matplotlib.gridspec as gridspec  # Mise en page avancée des sous-graphiques.
import seaborn as sns         # Visualisations statistiques basées sur matplotlib.
import plotly.graph_objects as go  # Graphiques interactifs avancés (gauge, waterfall).
import plotly.express as px   # API rapide pour graphiques interactifs courants.
from plotly.subplots import make_subplots  # Sous-graphiques avec axes multiples en Plotly.
import warnings               # Suppression des avertissements non critiques.
import os                     # Gestion des fichiers et dossiers système.

# ── Configuration globale ────────────────────────────────────────────────────
warnings.filterwarnings('ignore')           # Évite les messages parasites pendant l'exécution.
pd.set_option('display.max_columns', None)  # Affiche toutes les colonnes sans troncature.
pd.set_option('display.float_format', '{:.2f}'.format)  # Arrondit les flottants à 2 décimales.
plt.rcParams.update({'font.family': 'DejaVu Sans', 'figure.facecolor': 'white',
                     'axes.spines.top': False, 'axes.spines.right': False})
# Applique un style propre et cohérent à tous les graphiques matplotlib.

os.makedirs('outputs', exist_ok=True)  # Crée le dossier de sortie pour sauvegarder les visuels.

# ── Palette de couleurs ───────────────────────────────────────────────────────
C = {
    'red':    '#C0392B',  # Zones critiques et pertes
    'orange': '#E67E22',  # Zones d'alerte
    'green':  '#27AE60',  # Zones rentables
    'blue':   '#2980B9',  # Données primaires
    'navy':   '#1A1A2E',  # Titres et textes importants
    'gold':   '#F39C12',  # Seuils et références
    'teal':   '#16A085',  # Kenya (marché prioritaire)
    'grey':   '#95A5A6',  # Éléments secondaires
}
# Dictionnaire centralisé pour garantir la cohérence visuelle sur tous les graphiques.

print('✅ Configuration chargée avec succès.')

✅ Configuration chargée avec succès.


---
## 1️⃣ Chargement & Audit des données

In [6]:
# ── Chargement des 5 fichiers CSV ────────────────────────────────────────────
agents   = pd.read_csv('agents.csv')    # 300 agents : id, région, expérience.
claims   = pd.read_csv('claims.csv')    # 1 200 sinistres : montant, date, statut.
clients  = pd.read_csv('clients.csv')   # 1 200 clients : âge, pays, date d'inscription.
policies = pd.read_csv('policies.csv')  # 1 200 contrats : produit, dates, agent.
premiums = pd.read_csv('premiums.csv')  # 1 200 primes : montant, date de paiement.

print('=== DIMENSIONS DES DATASETS ===')
for name, df in [('agents',agents),('claims',claims),('clients',clients),
                 ('policies',policies),('premiums',premiums)]:
    print(f'{name:<10} : {df.shape[0]:>5} lignes × {df.shape[1]} colonnes')
    # Affiche la taille de chaque fichier pour vérifier l'intégrité du chargement.

=== DIMENSIONS DES DATASETS ===
agents     :   300 lignes × 3 colonnes
claims     :  1200 lignes × 5 colonnes
clients    :  1200 lignes × 4 colonnes
policies   :  1200 lignes × 6 colonnes
premiums   :  1200 lignes × 4 colonnes


In [7]:
# ── Fonction d'audit rapide ───────────────────────────────────────────────────
def audit_df(df, name):
    """Affiche un rapport d'audit complet pour un DataFrame."""
   
    print(f'  AUDIT : {name.upper()}')
    print(f'{"="*55}')
    print(f'Shape     : {df.shape}')
    # Dimensions du dataset — vérifie que le fichier est complet.
    print(f'Doublons  : {df.duplicated().sum()}')
    # Nombre de lignes identiques — doivent être supprimées avant analyse.
    print(f'\nValeurs manquantes :')
    missing = df.isnull().sum()
    print(missing[missing > 0] if missing.sum() > 0 else '  Aucune valeur manquante ✅')
    # Identifie les colonnes avec données absentes — impacte les calculs d'agrégation.
    print(f'\nTypes :')
    print(df.dtypes)
    # Vérifie que les dates, nombres et textes sont dans le bon format.
    print(f'\nAperçu (3 premières lignes) :')
    print(df.head(3).to_string())

# Audit de chaque dataset
for name, df in [('agents',agents),('claims',claims),('clients',clients),
                 ('policies',policies),('premiums',premiums)]:
    audit_df(df, name)

  AUDIT : AGENTS
Shape     : (300, 3)
Doublons  : 0

Valeurs manquantes :
  Aucune valeur manquante ✅

Types :
agent_id            int64
region                str
experience_years    int64
dtype: object

Aperçu (3 premières lignes) :
   agent_id         region  experience_years
0         1        Sénégal                 6
1         2          Maroc                11
2         3  Côte d’Ivoire                10
  AUDIT : CLAIMS
Shape     : (1200, 5)
Doublons  : 0

Valeurs manquantes :
  Aucune valeur manquante ✅

Types :
claim_id          int64
policy_id         int64
claim_amount    float64
claim_date          str
status              str
dtype: object

Aperçu (3 premières lignes) :
   claim_id  policy_id  claim_amount  claim_date    status
0         1        804       2126.54  2020-05-25  Accepted
1         2        220       3020.36  2023-10-07  Accepted
2         3         15        272.70  2024-12-28  Rejected
  AUDIT : CLIENTS
Shape     : (1200, 4)
Doublons  : 0

Valeurs manquantes

In [8]:
# ── Valeurs uniques des colonnes catégorielles ───────────────────────────────
print('=== VALEURS UNIQUES PAR COLONNE CATÉGORIELLE ===')

print(f'\nPays clients     : {sorted(clients["country"].unique())}')
# Liste les marchés géographiques — vérifie l'absence de doublons orthographiques.

print(f'Produits         : {list(policies["product_type"].unique())}')
# Liste les 4 lignes de produits — vérifie la cohérence des libellés.

print(f'Statuts sinistres: {list(claims["status"].unique())}')
# CRITIQUE : les statuts sont sensibles à la casse. Ici : "Accepted" et "Rejected".

print(f'Régions agents   : {sorted(agents["region"].unique())}')
# Vérifie l'alignement entre régions agents et pays clients.

print(f"\n⚠️ ATTENTION : Le statut 'Accepted' est avec majuscule.")
print(f"   Toujours filtrer avec : claims['status'] == 'Accepted'")
print(f"   Le filtre == 'accepted' (minuscule) retourne 0 résultat.")
# Avertissement critique documenté — erreur fréquente qui fausse tous les KPIs.

=== VALEURS UNIQUES PAR COLONNE CATÉGORIELLE ===

Pays clients     : ['Côte d’Ivoire', 'Kenya', 'Maroc', 'Nigeria', 'Sénégal']
Produits         : ['Santé', 'Habitation', 'Vie', 'Auto']
Statuts sinistres: ['Accepted', 'Rejected']
Régions agents   : ['Côte d’Ivoire', 'Kenya', 'Maroc', 'Nigeria', 'Sénégal']

⚠️ ATTENTION : Le statut 'Accepted' est avec majuscule.
   Toujours filtrer avec : claims['status'] == 'Accepted'
   Le filtre == 'accepted' (minuscule) retourne 0 résultat.


---
## 2️⃣ Nettoyage & Préparation

In [9]:
# ── Conversion des colonnes de dates ─────────────────────────────────────────
policies['start_date']    = pd.to_datetime(policies['start_date'])   # Date de début de contrat.
policies['end_date']      = pd.to_datetime(policies['end_date'])     # Date de fin de contrat.
claims['claim_date']      = pd.to_datetime(claims['claim_date'])     # Date de déclaration du sinistre.
premiums['payment_date']  = pd.to_datetime(premiums['payment_date']) # Date de paiement de la prime.
clients['signup_date']    = pd.to_datetime(clients['signup_date'])   # Date d'inscription du client.
# Toutes les colonnes texte contenant des dates sont converties en format datetime.
# Indispensable pour les calculs de durée et les analyses temporelles.

# ── Détection des contrats incohérents ───────────────────────────────────────
incoherent = policies[policies['start_date'] > policies['end_date']]
# Identifie les contrats où la date de début est postérieure à la date de fin — erreur de saisie.
print(f'Contrats avec start_date > end_date : {len(incoherent)} ({len(incoherent)/len(policies)*100:.1f}%)')
# Ces 403 contrats ont des durées négatives si non corrigées.

# ── Calcul de la durée contrat (valeur absolue pour corriger les inversions) ─
policies['duree_contrat_jours'] = (policies['end_date'] - policies['start_date']).dt.days.abs()
# abs() corrige les durées négatives dues aux dates inversées sans supprimer les données.
# Principe : on ne peut pas deviner la vraie date, on neutralise l'erreur proprement.

# ── Extraction des variables temporelles ─────────────────────────────────────
policies['annee_debut']    = policies['start_date'].dt.year   # Année de souscription du contrat.
policies['mois_debut']     = policies['start_date'].dt.month  # Mois de souscription — analyse saisonnière.
claims['annee_sinistre']   = claims['claim_date'].dt.year     # Année du sinistre — analyse P&L annuel.
premiums['annee_paiement'] = premiums['payment_date'].dt.year # Année de paiement — analyse revenus.
clients['annee_signup']    = clients['signup_date'].dt.year   # Année d'inscription — analyse cohortes.

# ── Vérification clients sans policies ───────────────────────────────────────
clients_sans_police = set(clients['client_id']) - set(policies['client_id'])
print(f'Clients sans aucun contrat : {len(clients_sans_police)}')
# Ces clients existent dans la base mais n'ont souscrit aucune police — à monitorer.

print('\n✅ Nettoyage terminé.')
print(f'Durée contrat moyenne : {policies["duree_contrat_jours"].mean():.0f} jours')
print(f'Plage dates sinistres : {claims["claim_date"].min().date()} → {claims["claim_date"].max().date()}')

Contrats avec start_date > end_date : 403 (33.6%)
Clients sans aucun contrat : 435

✅ Nettoyage terminé.
Durée contrat moyenne : 796 jours
Plage dates sinistres : 2020-01-02 → 2025-12-31


---
## 3️⃣ Construction du Dataset Maître (df_master)

In [10]:
# ── Agrégation des primes par police ─────────────────────────────────────────
premiums_agg = (
    premiums
    .groupby('policy_id')
    .agg(total_premium=('amount', 'sum'))  # Somme de toutes les primes encaissées par contrat.
    .reset_index()                          # Convertit l'index en colonne pour la jointure.
)
# Une police peut avoir plusieurs paiements de prime (mensuel, trimestriel).
# On agrège en une seule ligne par police pour simplifier les jointures.

# ── Agrégation de TOUS les sinistres par police ───────────────────────────────
claims_all = (
    claims
    .groupby('policy_id')
    .agg(
        total_claim   = ('claim_amount', 'sum'),  # Montant total de tous les sinistres déclarés.
        nb_claims     = ('claim_id',    'count')  # Nombre total de sinistres déclarés.
    )
    .reset_index()
)

# ── Agrégation des sinistres ACCEPTÉS uniquement ──────────────────────────────
# ⚠️ CRITIQUE : 'Accepted' avec majuscule — le filtre 'accepted' retourne 0 ligne
claims_acc = (
    claims[claims['status'] == 'Accepted']  # Filtre strict sur la casse exacte du fichier.
    .groupby('policy_id')
    .agg(
        total_claim_accepted = ('claim_amount', 'sum'),   # Montant réellement payé par l'assureur.
        nb_claims_accepted   = ('claim_id',    'count')   # Nombre de sinistres ayant donné lieu à paiement.
    )
    .reset_index()
)

# ── Jointure principale — construction du dataset maître ─────────────────────
df_master = policies.copy()                          # Base : une ligne = un contrat.

df_master = df_master.merge(clients,       on='client_id',  how='left')  # Profil client (âge, pays).
df_master = df_master.merge(agents,        on='agent_id',   how='left')  # Données agent (région, expérience).
df_master = df_master.merge(premiums_agg,  on='policy_id',  how='left')  # Revenus par contrat.
df_master = df_master.merge(claims_all,    on='policy_id',  how='left')  # Tous les sinistres.
df_master = df_master.merge(claims_acc,    on='policy_id',  how='left')  # Sinistres acceptés uniquement.
# Jointures LEFT : on conserve tous les contrats même sans sinistre ou sans prime enregistrée.

# ── Remplissage des NaN financiers ───────────────────────────────────────────
for col in ['total_premium','total_claim','total_claim_accepted','nb_claims','nb_claims_accepted']:
    df_master[col] = df_master[col].fillna(0)
# Un contrat sans sinistre a naturellement 0 — le NaN traduit une absence, pas une donnée manquante.

# ── Calcul des KPIs financiers par contrat ────────────────────────────────────
df_master['loss_ratio'] = np.where(
    df_master['total_premium'] > 0,
    df_master['total_claim_accepted'] / df_master['total_premium'] * 100,
    0  # Évite la division par zéro pour les contrats sans prime.
)
# Loss Ratio = (sinistres payés / primes encaissées) × 100. Seuil sain : < 80%.

df_master['profit'] = df_master['total_premium'] - df_master['total_claim_accepted']
# Profit technique par contrat — peut être négatif si les sinistres dépassent les primes.

df_master['is_profitable'] = df_master['profit'] > 0
# Variable binaire : True si le contrat génère un profit, False sinon.

# ── Sélection finale des colonnes ────────────────────────────────────────────
df_master = df_master[[
    'policy_id','client_id','product_type','start_date','end_date',
    'agent_id','region','experience_years',
    'age','country','signup_date','annee_signup',
    'total_premium','total_claim','total_claim_accepted',
    'nb_claims','nb_claims_accepted',
    'loss_ratio','profit','is_profitable',
    'duree_contrat_jours','annee_debut','mois_debut'
]]

print(f'✅ df_master construit : {df_master.shape[0]:,} lignes × {df_master.shape[1]} colonnes')
print(f'Colonnes : {list(df_master.columns)}')
df_master.head(5)

✅ df_master construit : 1,200 lignes × 23 colonnes
Colonnes : ['policy_id', 'client_id', 'product_type', 'start_date', 'end_date', 'agent_id', 'region', 'experience_years', 'age', 'country', 'signup_date', 'annee_signup', 'total_premium', 'total_claim', 'total_claim_accepted', 'nb_claims', 'nb_claims_accepted', 'loss_ratio', 'profit', 'is_profitable', 'duree_contrat_jours', 'annee_debut', 'mois_debut']


,policy_id,client_id,product_type,start_date,end_date,agent_id,region,experience_years,age,country,signup_date,annee_signup,total_premium,total_claim,total_claim_accepted,nb_claims,nb_claims_accepted,loss_ratio,profit,is_profitable,duree_contrat_jours,annee_debut,mois_debut
0,1,137,Santé,2020-02-25,2026-12-03,109,Kenya,4,44,Côte d’Ivoire,2020-04-28,2020,0.00,3282.07,3282.07,1.00,1.00,0.00,-3282.07,False,2473,2020,2
1,2,1199,Santé,2021-12-20,2021-01-31,168,Nigeria,7,64,Nigeria,2024-01-01,2024,0.00,0.00,0.00,0.00,0.00,0.00,0.00,False,323,2021,12
2,3,958,Habitation,2025-09-10,2024-09-28,300,Maroc,11,43,Sénégal,2020-02-22,2020,291.53,0.00,0.00,0.00,0.00,0.00,291.53,True,347,2025,9
3,4,19,Vie,2021-07-27,2024-11-01,218,Sénégal,14,19,Nigeria,2021-11-20,2021,0.00,2182.24,2182.24,1.00,1.00,0.00,-2182.24,False,1193,2021,7
4,5,195,Auto,2021-10-28,2024-02-28,222,Kenya,14,44,Sénégal,2025-01-04,2025,110.70,0.00,0.00,0.00,0.00,0.00,110.70,True,853,2021,10


---
## 4️⃣ KPIs Globaux

In [11]:
# ── Calcul des KPIs globaux ───────────────────────────────────────────────────
ca_total     = df_master['total_premium'].sum()           # Total des primes encaissées sur 6 ans.
sin_total    = df_master['total_claim_accepted'].sum()    # Total des sinistres payés sur 6 ans.
profit_net   = df_master['profit'].sum()                  # Résultat technique global = primes - sinistres.
lr_global    = sin_total / ca_total * 100                 # Loss Ratio global — indicateur de rentabilité.
panier_moyen = df_master['total_premium'].mean()          # Prime moyenne par contrat.
taux_accept  = (df_master['nb_claims_accepted'].sum() /   # Proportion de sinistres ayant donné lieu à paiement.
                df_master['nb_claims'].sum() * 100)
n_clients    = df_master['client_id'].nunique()           # Nombre de clients distincts dans le portefeuille.
n_contrats   = len(df_master)                             # Nombre total de contrats analysés.
pct_rentable = df_master['is_profitable'].mean() * 100    # Part des contrats générant un profit positif.

# ── Affichage structuré ───────────────────────────────────────────────────────

print('  PERFORMANCE GLOBALE — COMPAGNIE D\'ASSURANCE AFRICAINE')
print('  Période : Janvier 2020 – Décembre 2025')

print(f'  Total primes encaissées  : {ca_total:>15,.2f}')
print(f'  Total sinistres acceptés : {sin_total:>15,.2f}')
print(f'  Perte nette cumulée      : {profit_net:>15,.2f}  🔴')
print(f'  Loss Ratio global        : {lr_global:>14.1f}%  🔴 (norme : < 80%)')
print(f'  Marge nette              : {profit_net/ca_total*100:>13.1f}%')
print(f'  Prime moyenne / contrat  : {panier_moyen:>15,.2f}')
print(f'  Taux d\'acceptation       : {taux_accept:>14.1f}%  ⚠️ (norme : 60–65%)')
print(f'  Clients uniques          : {n_clients:>15,}')
print(f'  Contrats totaux          : {n_contrats:>15,}')
print(f'  Contrats rentables       : {df_master["is_profitable"].sum():>6} / {n_contrats} ({pct_rentable:.1f}%)')


  PERFORMANCE GLOBALE — COMPAGNIE D'ASSURANCE AFRICAINE
  Période : Janvier 2020 – Décembre 2025
  Total primes encaissées  :    1,246,153.84
  Total sinistres acceptés :    2,359,188.37
  Perte nette cumulée      :   -1,113,034.53  🔴
  Loss Ratio global        :          189.3%  🔴 (norme : < 80%)
  Marge nette              :         -89.3%
  Prime moyenne / contrat  :        1,038.46
  Taux d'acceptation       :           80.9%  ⚠️ (norme : 60–65%)
  Clients uniques          :             765
  Contrats totaux          :           1,200
  Contrats rentables       :    452 / 1200 (37.7%)


---
## 5️⃣ Analyse par Produit

In [12]:
# ── Agrégation par produit ────────────────────────────────────────────────────
prod = df_master.groupby('product_type').agg(
    Nb_contrats      = ('policy_id',              'count'),  # Volume du portefeuille par produit.
    Primes           = ('total_premium',           'sum'),   # Revenus générés par produit.
    Sinistres        = ('total_claim_accepted',    'sum'),   # Sinistres payés par produit.
    Profit           = ('profit',                  'sum'),   # Résultat technique par produit.
    Nb_sinistres     = ('nb_claims_accepted',      'sum'),   # Volume de sinistres par produit.
).reset_index()
# Groupby sur product_type pour obtenir la performance financière de chaque ligne de produit.

prod['Loss_Ratio_%'] = (prod['Sinistres'] / prod['Primes'] * 100).round(1)
# Loss Ratio = proportion du revenu consommée par les sinistres — indicateur de rentabilité produit.

prod['Profit_%_CA']  = (prod['Profit'] / prod['Primes'] * 100).round(1)
# Marge nette en % du CA — indique combien on garde pour 100 encaissés.

prod['Frequence']    = (prod['Nb_sinistres'] / prod['Nb_contrats']).round(3)
# Fréquence de sinistralité = nombre moyen de sinistres par contrat — clé pour la tarification.

prod['Cout_moyen']   = np.where(prod['Nb_sinistres'] > 0,
                                 prod['Sinistres'] / prod['Nb_sinistres'], 0).round(2)
# Coût moyen par sinistre — avec protection contre division par zéro.

prod = prod.sort_values('Loss_Ratio_%', ascending=False)
# Tri décroissant : les produits les plus destructeurs apparaissent en premier.

print('ANALYSE PAR PRODUIT')
print('=' * 80)
print(prod[['product_type','Nb_contrats','Primes','Sinistres','Profit',
            'Loss_Ratio_%','Profit_%_CA','Frequence','Cout_moyen']].to_string(index=False))

print('\n⚠️ Tous les produits sont déficitaires.')
print(f'  Habitation : LR {prod[prod["product_type"]=="Habitation"]["Loss_Ratio_%"].values[0]}% — produit le plus critique.')
print(f'  Santé      : LR {prod[prod["product_type"]=="Santé"]["Loss_Ratio_%"].values[0]}% — produit à prioriser pour révision tarifaire.')

ANALYSE PAR PRODUIT
product_type  Nb_contrats    Primes  Sinistres     Profit  Loss_Ratio_%  Profit_%_CA  Frequence  Cout_moyen
  Habitation          323 345005.82  722471.57 -377465.75        209.40      -109.40       0.90     2491.28
         Vie          269 267026.84  553621.89 -286595.05        207.30      -107.30       0.87     2376.06
        Auto          299 287564.48  510542.49 -222978.01        177.50       -77.50       0.73     2352.73
       Santé          309 346556.70  572552.42 -225995.72        165.20       -65.20       0.75     2478.58

⚠️ Tous les produits sont déficitaires.
  Habitation : LR 209.4% — produit le plus critique.
  Santé      : LR 165.2% — produit à prioriser pour révision tarifaire.


---
## 6️⃣ Analyse Géographique

In [13]:
# ── Agrégation par pays ───────────────────────────────────────────────────────
geo = df_master.groupby('country').agg(
    Nb          = ('policy_id',           'count'),  # Volume de contrats par marché.
    Primes      = ('total_premium',        'sum'),   # Revenus par pays.
    Sinistres   = ('total_claim_accepted', 'sum'),   # Sinistres payés par pays.
    Profit      = ('profit',               'sum'),   # Résultat technique par marché.
).reset_index()
# Analyse géographique pour identifier les marchés à développer vs à réduire.

geo['LR_%']      = (geo['Sinistres'] / geo['Primes'] * 100).round(1)
# Loss Ratio par pays — mesure l'intensité du risque selon le marché.

geo['Part_CA_%'] = (geo['Primes'] / geo['Primes'].sum() * 100).round(1)
# Part du CA total — indique le poids relatif de chaque marché dans le portefeuille.

geo['Perte_unit'] = (geo['Profit'] / geo['Nb']).round(0)
# Perte moyenne par contrat par pays — normalise pour comparer des marchés de tailles différentes.

geo = geo.sort_values('LR_%', ascending=False)

print('ANALYSE PAR PAYS')
print('=' * 80)
print(geo[['country','Nb','Primes','Sinistres','Profit','LR_%','Part_CA_%','Perte_unit']].to_string(index=False))

# Analyse croisée pays × produit
print('\nHEATMAP LOSS RATIO PAR PAYS × PRODUIT :')
cross = df_master.groupby(['country','product_type']).agg(
    LR=('loss_ratio','mean')).reset_index()
pivot = cross.pivot(index='country', columns='product_type', values='LR').round(1)
print(pivot.to_string())
print('\n🟢 Kenya = seul marché avec LR < 160% — marché prioritaire pour expansion.')
print('🔴 Maroc = LR le plus élevé — gel des nouvelles souscriptions recommandé.')

ANALYSE PAR PAYS
      country  Nb    Primes  Sinistres     Profit   LR_%  Part_CA_%  Perte_unit
        Maroc 223 214252.86  443513.04 -229260.18 207.00      17.20    -1028.00
      Sénégal 260 260587.63  519173.88 -258586.25 199.20      20.90     -995.00
Côte d’Ivoire 240 272741.45  532847.64 -260106.19 195.40      21.90    -1084.00
      Nigeria 239 240840.70  466035.48 -225194.78 193.50      19.30     -942.00
        Kenya 238 257731.20  397618.33 -139887.13 154.30      20.70     -588.00

HEATMAP LOSS RATIO PAR PAYS × PRODUIT :
product_type    Auto  Habitation  Santé    Vie
country                                       
Côte d’Ivoire 154.20      165.60 161.80 113.70
Kenya         135.10       93.80  95.00  68.10
Maroc         134.80      158.80 141.10 122.20
Nigeria       160.10      119.60  64.70 156.40
Sénégal        84.70      149.70 122.80 200.00

🟢 Kenya = seul marché avec LR < 160% — marché prioritaire pour expansion.
🔴 Maroc = LR le plus élevé — gel des nouvelles souscriptio

---
## 7️⃣ Analyse Temporelle

In [14]:
# ── Évolution annuelle ────────────────────────────────────────────────────────
yr = df_master.groupby('annee_debut').agg(
    Nb_contrats = ('policy_id',           'count'),  # Volume de souscriptions par année.
    Primes      = ('total_premium',        'sum'),   # Revenus par année de souscription.
    Sinistres   = ('total_claim_accepted', 'sum'),   # Sinistres payés par année.
    Profit      = ('profit',               'sum'),   # Résultat technique annuel.
).reset_index()
# Analyse temporelle basée sur l'année de début de contrat — vision portefeuille.

yr['LR_%']        = (yr['Sinistres'] / yr['Primes'] * 100).round(1)
# Loss Ratio annuel — permet de suivre l'évolution de la rentabilité dans le temps.

yr['Croissance_%'] = yr['Primes'].pct_change().mul(100).round(1)
# Variation des primes d'une année sur l'autre — mesure la croissance du portefeuille.

yr['Signal'] = yr['LR_%'].apply(
    lambda x: '🔴 CRITIQUE' if x > 200 else '🟠 GRAVE' if x > 180 else '🟡 ÉLEVÉ'
)
# Classification automatique de la sévérité par année — aide la lecture rapide.

print('ÉVOLUTION ANNUELLE 2020–2025')
print('=' * 75)
print(yr[['annee_debut','Nb_contrats','Primes','Sinistres','Profit','LR_%','Croissance_%','Signal']].to_string(index=False))

print('\n📌 2021 : Meilleure année (LR 161%) — fenêtre d\'opportunité manquée.')
print('📌 2022 : Rupture (LR 209%) — aggravation structurelle débutée.')
print('📌 2024 : Pire ratio (LR 222%) + chute des primes de 26% — signal d\'alerte maximal.')

ÉVOLUTION ANNUELLE 2020–2025
 annee_debut  Nb_contrats    Primes  Sinistres     Profit   LR_%  Croissance_%     Signal
        2020          213 225537.95  422133.30 -196595.35 187.20           NaN    🟠 GRAVE
        2021          203 226803.24  365090.24 -138287.00 161.00          0.60    🟡 ÉLEVÉ
        2022          214 207600.62  433330.53 -225729.91 208.70         -8.50 🔴 CRITIQUE
        2023          209 224447.98  405086.19 -180638.21 180.50          8.10    🟠 GRAVE
        2024          176 166042.92  367925.48 -201882.56 221.60        -26.00 🔴 CRITIQUE
        2025          185 195721.13  365622.63 -169901.50 186.80         17.90    🟠 GRAVE

📌 2021 : Meilleure année (LR 161%) — fenêtre d'opportunité manquée.
📌 2022 : Rupture (LR 209%) — aggravation structurelle débutée.
📌 2024 : Pire ratio (LR 222%) + chute des primes de 26% — signal d'alerte maximal.


---
## 8️⃣ Gestion du Risque

In [15]:
# ── Fréquence et coût moyen des sinistres ────────────────────────────────────
risk = df_master.groupby('product_type').agg(
    N_polices         = ('policy_id',             'count'),   # Nombre de contrats en portefeuille.
    N_sin_acceptes    = ('nb_claims_accepted',    'sum'),     # Sinistres ayant donné lieu à paiement.
    N_sin_total       = ('nb_claims',             'sum'),     # Tous les sinistres déclarés.
    Cout_total_sin    = ('total_claim_accepted',  'sum'),     # Coût total des sinistres acceptés.
).reset_index()

risk['Frequence']     = (risk['N_sin_acceptes'] / risk['N_polices']).round(3)
# Fréquence = sinistres acceptés / contrats. Proche de 1 = quasi un sinistre par contrat.

risk['Cout_moyen']    = np.where(risk['N_sin_acceptes'] > 0,
    risk['Cout_total_sin'] / risk['N_sin_acceptes'], 0).round(2)
# Coût moyen par sinistre = severity. La prime doit couvrir fréquence × severity.

risk['Risk_cost']     = (risk['Cout_total_sin'] / risk['N_polices']).round(2)
# Coût du risque par police = prime pure nécessaire pour équilibrer les sinistres.

risk['Taux_accept_%'] = (risk['N_sin_acceptes'] / risk['N_sin_total'] * 100).round(1)
# Taux d'acceptation par produit — norme sectorielle : 60–65%.

risk = risk.sort_values('Frequence', ascending=False)

print('ANALYSE DU RISQUE PAR PRODUIT')
print('=' * 80)
print(risk[['product_type','N_polices','Frequence','Cout_moyen','Risk_cost','Taux_accept_%']].to_string(index=False))

print('\n📌 Habitation : fréquence 0.898 = quasi un sinistre par contrat.')
print('📌 Taux acceptation global 80.9% vs norme 60–65% → critère d\'instruction à renforcer.')

# Distribution des montants de sinistres
print('\nDISTRIBUTION DES MONTANTS DE SINISTRES :')
print(claims[claims['status']=='Accepted']['claim_amount'].describe().round(2))
# Statistiques descriptives des sinistres acceptés — identifie les sinistres extrêmes.

ANALYSE DU RISQUE PAR PRODUIT
product_type  N_polices  Frequence  Cout_moyen  Risk_cost  Taux_accept_%
  Habitation        323       0.90     2491.28    2236.75          82.90
         Vie        269       0.87     2376.06    2058.07          81.80
       Santé        309       0.75     2478.58    1852.92          77.80
        Auto        299       0.73     2352.73    1707.50          81.00

📌 Habitation : fréquence 0.898 = quasi un sinistre par contrat.
📌 Taux acceptation global 80.9% vs norme 60–65% → critère d'instruction à renforcer.

DISTRIBUTION DES MONTANTS DE SINISTRES :
count    971.00
mean    2429.65
std     1429.34
min        6.02
25%     1185.02
50%     2379.46
75%     3616.97
max     4991.52
Name: claim_amount, dtype: float64


---
## 9️⃣ Analyse Clients & CLV

In [16]:
# ── Customer Lifetime Value (CLV technique) ───────────────────────────────────
clv = df_master.groupby('client_id').agg(
    Nb_contrats = ('policy_id',           'count'),  # Nombre de contrats souscrits par client.
    Primes      = ('total_premium',        'sum'),   # Total des primes payées par le client.
    Sinistres   = ('total_claim_accepted', 'sum'),   # Total des sinistres reçus par le client.
    CLV         = ('profit',               'sum'),   # CLV technique = primes - sinistres.
).reset_index()
# CLV simplifiée : valeur nette générée par le client sur toute la durée de la relation.
# Ne tient pas compte des frais d'acquisition et d'administration (données non disponibles).

clv['LR_%'] = np.where(clv['Primes'] > 0, clv['Sinistres'] / clv['Primes'] * 100, 0).round(1)
# Loss Ratio individuel — identifie les clients systématiquement sinistrogènes.

clv['Segment'] = pd.cut(clv['CLV'],
    bins  = [-np.inf, -1000, 0, 500, np.inf],
    labels= ['Très déficitaire', 'Déficitaire', 'Neutre', 'Profitable']
)
# Segmentation en 4 groupes selon la CLV — guide les actions de rétention et de retarification.

# ── Métriques de segmentation ────────────────────────────────────────────────
n_clients_total   = len(clv)
n_rentables       = (clv['CLV'] > 0).sum()
n20               = int(n_clients_total * 0.20)
top20_clv         = clv.nlargest(n20, 'CLV')['CLV'].sum()
bot10_clv         = clv.nsmallest(int(n_clients_total*0.10), 'CLV')['CLV'].sum()
top10_clv         = clv.nlargest(int(n_clients_total*0.10), 'CLV')['CLV'].sum()

print('ANALYSE CLIENTS & CLV')
print('=' * 55)
print(f'  Clients totaux               : {n_clients_total}')
print(f'  Clients rentables (CLV > 0)  : {n_rentables} ({n_rentables/n_clients_total*100:.1f}%)')
print(f'  CA top 20% clients           : {top20_clv:,.0f} ({top20_clv/clv["CLV"].sum()*100:.1f}%)')
print(f'  Profit top 10% clients       : {top10_clv:,.0f}')
print(f'  Perte bottom 10% clients     : {bot10_clv:,.0f}')
print(f'  Ratio destruction/création   : {abs(bot10_clv/top10_clv):.2f}x')

print('\nSEGMENTATION CLV :')
seg_stats = clv.groupby('Segment', observed=True).agg(
    Nb=('client_id','count'), CLV_total=('CLV','sum'), CLV_moyen=('CLV','mean')
).round(0)
seg_stats['Part_%'] = (seg_stats['Nb'] / n_clients_total * 100).round(1)
print(seg_stats.to_string())

print('\nTOP 10 CLIENTS PAR CLV :')
top10_clients = clv.nlargest(10, 'CLV')[['client_id','Nb_contrats','Primes','Sinistres','CLV','LR_%']]
print(top10_clients.to_string(index=False))

print('\nBOTTOM 10 CLIENTS (plus destructeurs) :')
bot10_clients = clv.nsmallest(10, 'CLV')[['client_id','Nb_contrats','Primes','Sinistres','CLV','LR_%']]
print(bot10_clients.to_string(index=False))

ANALYSE CLIENTS & CLV
  Clients totaux               : 765
  Clients rentables (CLV > 0)  : 274 (35.8%)
  CA top 20% clients           : 394,483 (-35.4%)
  Profit top 10% clients       : 256,807
  Perte bottom 10% clients     : -680,543
  Ratio destruction/création   : 2.65x

SEGMENTATION CLV :
                   Nb   CLV_total  CLV_moyen  Part_%
Segment                                             
Très déficitaire  334 -1556376.00   -4660.00   43.70
Déficitaire       157   -32467.00    -207.00   20.50
Neutre             48    13506.00     281.00    6.30
Profitable        226   462302.00    2046.00   29.50

TOP 10 CLIENTS PAR CLV :
 client_id  Nb_contrats  Primes  Sinistres     CLV  LR_%
       556            2 6864.57       0.00 6864.57  0.00
      1046            1 5389.69       0.00 5389.69  0.00
      1122            2 5344.94       0.00 5344.94  0.00
      1032            1 7164.66    1963.75 5200.91 27.40
       804            2 7136.33    2022.58 5113.75 28.30
       720        

---
## 🔟 Performance des Agents

In [17]:
# ── Agrégation par agent ──────────────────────────────────────────────────────
ag = df_master.groupby(['agent_id','region','experience_years']).agg(
    Nb_contrats = ('policy_id',           'count'),  # Volume du portefeuille géré par l'agent.
    Primes      = ('total_premium',        'sum'),   # Revenus générés par l'agent.
    Sinistres   = ('total_claim_accepted', 'sum'),   # Sinistres dans le portefeuille de l'agent.
    Profit      = ('profit',               'sum'),   # Résultat technique du portefeuille de l'agent.
).reset_index()
# Un agent performant génère des primes ET sélectionne de bons risques (faible sinistralité).

ag['LR_%'] = np.where(ag['Primes'] > 0,
    ag['Sinistres'] / ag['Primes'] * 100, np.inf).round(1)
# Loss Ratio par agent — un agent avec LR > 200% détruit plus de valeur qu'il n'en crée.
# np.inf pour les agents avec primes = 0 (anomalie critique à investiguer).

ag['Perf_label'] = ag['Profit'].apply(
    lambda x: '🟢 Rentable' if x > 0 else '🔴 Déficitaire'
)
# Label de performance binaire pour faciliter la lecture rapide des classements.

print('TOP 10 AGENTS PAR PROFIT')
print('=' * 75)
print(ag.nlargest(10,'Profit')[['agent_id','region','experience_years',
                                  'Nb_contrats','Primes','Profit','LR_%']].to_string(index=False))

print('\nBOTTOM 10 AGENTS (plus destructeurs)')
print('=' * 75)
print(ag.nsmallest(10,'Profit')[['agent_id','region','experience_years',
                                   'Nb_contrats','Primes','Sinistres','Profit','LR_%']].to_string(index=False))

print('\n⚠️ ANOMALIE CRITIQUE — Agent 127 :')
agent_127 = ag[ag['agent_id'] == 127]
print(agent_127.to_string(index=False))
print('   → Sinistres sans primes enregistrées = impossibilité technique.')
print('   → Hypothèses : erreur de saisie, manipulation ou fraude à la souscription.')
print('   → Action : audit manuel immédiat avant toute autre décision.')

# Corrélation expérience vs performance
corr = ag[['experience_years','Profit']].corr().iloc[0,1]
print(f'\n📊 Corrélation expérience vs profit : {corr:.3f}')
print('   → Corrélation proche de 0 : l\'expérience ne prédit pas la performance.')
# Une corrélation nulle confirme que les meilleurs agents ont 1–3 ans d'expérience.

TOP 10 AGENTS PAR PROFIT
 agent_id  region  experience_years  Nb_contrats   Primes  Profit  LR_%
      234 Nigeria                 3           10 15903.16 7678.18 51.70
      229 Nigeria                 1            5  9220.49 6944.81 24.70
       33   Kenya                12            6  8362.39 6827.70 18.40
       68 Nigeria                11            4  6859.60 6771.77  1.30
      113   Kenya                14            5  6848.62 6625.62  3.30
      163 Nigeria                10            9 12302.11 6504.33 47.10
      173 Nigeria                 4            8 19595.89 6403.72 67.30
      286   Maroc                 9            3  8870.78 6221.28 29.90
       73   Maroc                 1            5  8664.09 6206.02 28.40
      110   Kenya                11            5  5925.74 5925.74  0.00

BOTTOM 10 AGENTS (plus destructeurs)
 agent_id        region  experience_years  Nb_contrats  Primes  Sinistres    Profit   LR_%
      127       Nigeria                 9            5

---
## 1️⃣1️⃣ Visualisations Stratégiques

In [19]:
# ── VIZ 1 : Gauge Loss Ratio global (Plotly) ─────────────────────────────────
lr_val = df_master['total_claim_accepted'].sum() / df_master['total_premium'].sum() * 100
# Calcule le Loss Ratio global depuis les données — jamais en dur dans le code.

fig1 = go.Figure(go.Indicator(
    mode  = 'gauge+number+delta',          # Affiche la jauge, la valeur et l'écart vs référence.
    value = round(lr_val, 1),              # Loss Ratio calculé dynamiquement depuis df_master.
    delta = {'reference': 100,             # Écart par rapport au seuil d'équilibre 100%.
             'increasing': {'color': '#C0392B'},   # Rouge si au-dessus de 100%.
             'decreasing': {'color': '#27AE60'}},  # Vert si en dessous de 100%.
    number = {'suffix': '%', 'font': {'size': 52, 'color': '#C0392B'}},
    title  = {'text': '<b>Loss Ratio Global</b><br><span style="font-size:13px;color:grey">Seuil sain : &lt; 80% · Seuil critique : 100%</span>'},
    gauge  = {
        'axis'  : {'range': [0, 250]},     # Plage étendue à 250% pour couvrir les cas extrêmes.
        'bar'   : {'color': '#1A1A2E', 'thickness': 0.25},
        'steps' : [
            {'range': [0, 80],   'color': '#D5F5E3'},   # Zone verte : rentable.
            {'range': [80, 100], 'color': '#FDEBD0'},   # Zone orange : limite.
            {'range': [100, 250],'color': '#FADBD8'},   # Zone rouge : critique.
        ],
        'threshold': {'line': {'color': '#C0392B', 'width': 4}, 'value': lr_val}
        # Marqueur rouge sur la valeur réelle pour la situer dans la jauge.
    }
))
fig1.update_layout(height=400, width=600, paper_bgcolor='white')
fig1.write_html('outputs/viz1_gauge_loss_ratio.html')  # Export interactif pour partage.
fig1.show()
print('✅ VIZ 1 — Gauge Loss Ratio sauvegardée.')

ValueError: Mime type rendering requires nbformat>=4.2.0 but it is not installed

In [ ]:
# ── VIZ 2 : Waterfall P&L par produit (Plotly) ───────────────────────────────
agg_prod = df_master.groupby('product_type').agg(
    Primes   = ('total_premium',        'sum'),  # Revenus par produit.
    Sinistres= ('total_claim_accepted', 'sum'),  # Sinistres par produit.
).reset_index()
agg_prod['Profit'] = agg_prod['Primes'] - agg_prod['Sinistres']  # Résultat net par produit.
agg_prod['LR_%']   = (agg_prod['Sinistres'] / agg_prod['Primes'] * 100).round(1)
agg_prod = agg_prod.sort_values('Profit')
# Tri croissant : les produits les plus déficitaires s'affichent en bas.

fig2 = go.Figure(go.Waterfall(
    name      = 'P&L par Produit',
    measure   = ['absolute'] * len(agg_prod),     # Chaque barre = valeur indépendante (pas cumulative).
    x         = agg_prod['product_type'],          # Axe X = les 4 produits.
    y         = agg_prod['Profit'],                # Axe Y = profit net (négatif = perte).
    text      = [f"LR: {lr}%" for lr in agg_prod['LR_%']],  # Annotation Loss Ratio sur chaque barre.
    textposition = 'outside',
    decreasing   = {'marker': {'color': '#C0392B'}},  # Rouge pour les pertes.
    increasing   = {'marker': {'color': '#27AE60'}},  # Vert pour les gains (aucun ici).
    connector    = {'line': {'color': 'grey', 'dash': 'dot'}}
))
fig2.update_layout(
    title  = '<b>Profit / Perte par Produit</b><br><span style="font-size:12px">Tous les produits sont déficitaires — Habitation détruit le plus de valeur</span>',
    height = 480, width = 800, paper_bgcolor = 'white',
    yaxis  = dict(title='Profit net', tickformat=',.0f')
)
fig2.write_html('outputs/viz2_waterfall_produit.html')  # Graphique interactif exporté.
fig2.show()
print('✅ VIZ 2 — Waterfall produit sauvegardée.')

In [ ]:
# ── VIZ 3 : Loss Ratio par pays (Plotly bar horizontal) ──────────────────────
geo_viz = df_master.groupby('country').agg(
    Primes   = ('total_premium',        'sum'),
    Sinistres= ('total_claim_accepted', 'sum'),
    Profit   = ('profit',               'sum')
).reset_index()
geo_viz['LR_%'] = (geo_viz['Sinistres'] / geo_viz['Primes'] * 100).round(1)
geo_viz = geo_viz.sort_values('LR_%', ascending=True)  # Croissant pour barh lisible.

bar_colors = [  # Code couleur selon la sévérité du Loss Ratio par pays.
    '#C0392B' if lr > 190 else '#E67E22' if lr > 170 else '#27AE60'
    for lr in geo_viz['LR_%']
]

fig3 = go.Figure(go.Bar(
    y           = geo_viz['country'],
    x           = geo_viz['LR_%'],
    orientation = 'h',                                        # Barres horizontales pour lecture du pays.
    marker_color= bar_colors,
    text        = [f"{lr}%  |  Perte: {abs(p):,.0f}" for lr, p in zip(geo_viz['LR_%'], geo_viz['Profit'])],
    textposition= 'outside'                                   # Annotations hors barre pour clarté.
))
fig3.add_vline(x=100, line_dash='dash', line_color='#F39C12',  # Ligne seuil d'équilibre.
               annotation_text='Seuil 100%', annotation_font_color='#F39C12')
fig3.add_vline(x=80, line_dash='dot', line_color='#27AE60',    # Ligne seuil optimal.
               annotation_text='Seuil optimal 80%', annotation_position='bottom right')
fig3.update_layout(
    title  = '<b>Loss Ratio par Pays</b><br><span style="font-size:12px">Kenya = seul marché à développer · Maroc = gel recommandé</span>',
    height = 420, width = 800, paper_bgcolor = 'white',
    xaxis  = dict(title='Loss Ratio (%)', range=[0, 260])
)
fig3.write_html('outputs/viz3_loss_ratio_pays.html')
fig3.show()
print('✅ VIZ 3 — Ranking géographique sauvegardé.')

In [ ]:
# ── VIZ 4 : Évolution temporelle 2020–2025 (double axe) ─────────────────────
trend = df_master.groupby('annee_debut').agg(
    Primes   = ('total_premium',        'sum'),
    Sinistres= ('total_claim_accepted', 'sum')
).reset_index().sort_values('annee_debut')
# Agrégation par année de souscription — vision évolution du portefeuille.

trend['LR_%'] = (trend['Sinistres'] / trend['Primes'] * 100).round(1)
# Loss Ratio annuel pour l'axe secondaire.

fig4 = make_subplots(specs=[[{'secondary_y': True}]])
# Double axe : montants à gauche, Loss Ratio en % à droite.

fig4.add_trace(go.Scatter(
    x=trend['annee_debut'], y=trend['Primes'],
    name='Primes encaissées', mode='lines+markers+text',
    line=dict(color='#2980B9', width=3), marker=dict(size=8),
    text=trend['Primes'].apply(lambda x: f"{x/1000:.0f}k"),
    textposition='top center'
), secondary_y=False)

fig4.add_trace(go.Scatter(
    x=trend['annee_debut'], y=trend['Sinistres'],
    name='Sinistres payés', mode='lines+markers+text',
    line=dict(color='#C0392B', width=3), marker=dict(size=8),
    text=trend['LR_%'].apply(lambda x: f"LR:{x:.0f}%"),
    textposition='bottom center'
), secondary_y=False)

fig4.add_trace(go.Scatter(
    x=trend['annee_debut'], y=trend['LR_%'],
    name='Loss Ratio (%)', mode='lines+markers',
    line=dict(color='#E67E22', width=2, dash='dot'), marker=dict(size=6)
), secondary_y=True)
# Courbe Loss Ratio sur axe secondaire pour ne pas écraser l'échelle des montants.

fig4.add_hline(y=100, secondary_y=True, line_dash='dash',
               line_color='#F39C12', line_width=1.5)  # Seuil LR 100% sur axe droit.

fig4.update_layout(
    title  = '<b>Évolution Annuelle 2020–2025</b><br><span style="font-size:12px">Sinistres dépassent systématiquement les primes — 2024 : pire ratio (222%)</span>',
    height = 480, width = 900, paper_bgcolor = 'white', legend=dict(orientation='h')
)
fig4.update_yaxes(title_text='Montants', secondary_y=False, tickformat=',.0f')
fig4.update_yaxes(title_text='Loss Ratio (%)', secondary_y=True)
fig4.write_html('outputs/viz4_evolution_temporelle.html')
fig4.show()
print('✅ VIZ 4 — Évolution temporelle sauvegardée.')

In [ ]:
# ── VIZ 5 : Scatter agents — Profit vs Expérience (Plotly) ───────────────────
ag_viz = df_master.groupby(['agent_id','region','experience_years']).agg(
    Nb     = ('policy_id',           'count'),
    Primes = ('total_premium',        'sum'),
    Sin    = ('total_claim_accepted', 'sum'),
    Profit = ('profit',               'sum')
).reset_index()
ag_viz['LR_%']  = np.where(ag_viz['Primes']>0, ag_viz['Sin']/ag_viz['Primes']*100, np.nan).round(1)
ag_viz['Statut'] = ag_viz['Profit'].apply(lambda x: 'Rentable' if x > 0 else 'Déficitaire')
# Segmentation binaire pour colorer les points selon la performance.

fig5 = px.scatter(
    ag_viz,
    x    = 'experience_years',        # Axe X : ancienneté de l'agent en années.
    y    = 'Profit',                  # Axe Y : profit généré par le portefeuille.
    color= 'Statut',
    color_discrete_map = {'Rentable': '#27AE60', 'Déficitaire': '#C0392B'},
    size = 'Nb',                      # Taille du point = volume du portefeuille.
    size_max = 25,
    symbol   = 'region',              # Forme = région pour distinguer les marchés.
    hover_data = {'agent_id': True, 'LR_%': True, 'Primes': ':.0f'},
    labels = {'experience_years': "Années d'expérience", 'Profit': 'Profit net'},
    title  = '<b>Performance Agents : Profit vs Expérience</b><br><span style="font-size:12px">Absence de corrélation — les meilleurs ont 1–3 ans</span>'
)
fig5.add_hline(y=0, line_dash='dash', line_color='#1A1A2E',  # Ligne de rentabilité.
               annotation_text='Seuil rentabilité')
fig5.update_layout(height=520, width=900, paper_bgcolor='white')
fig5.write_html('outputs/viz5_agents_scatter.html')
fig5.show()
print('✅ VIZ 5 — Scatter agents sauvegardé.')

In [ ]:
# ── VIZ 6 : Top 10 vs Bottom 10 agents (Plotly bar horizontal) ───────────────
top10_ag  = ag_viz.nlargest(10, 'Profit').copy()
bot10_ag  = ag_viz.nsmallest(10, 'Profit').copy()
# Sélection des 10 meilleurs et 10 pires agents par profit net.

top10_ag['Groupe'] = '🟢 Top 10'
bot10_ag['Groupe'] = '🔴 Bottom 10'
combined = pd.concat([bot10_ag, top10_ag]).reset_index(drop=True)
# Concaténation des deux groupes — bottom en premier pour ordre visuel logique.

combined['label']  = combined.apply(
    lambda r: f"Agent {r['agent_id']} | {r['region'][:3]} | {r['experience_years']}ans", axis=1
)
# Label compact pour chaque agent sur l'axe Y — économise l'espace.

combined['couleur'] = combined['Profit'].apply(
    lambda x: '#27AE60' if x > 0 else '#C0392B'
)
# Couleur verte/rouge selon la performance — lecture immédiate.

fig6 = go.Figure(go.Bar(
    y           = combined['label'],
    x           = combined['Profit'],
    orientation = 'h',
    marker_color= combined['couleur'],
    text        = combined['Profit'].apply(lambda x: f"{x:+,.0f}"),
    textposition= 'outside',
    customdata  = combined[['LR_%','Nb']],
    hovertemplate = '<b>%{y}</b><br>Profit: %{x:,.0f}<br>LR: %{customdata[0]:.1f}%<br>Contrats: %{customdata[1]}<extra></extra>'
))
fig6.add_vline(x=0, line_color='#1A1A2E', line_width=2)
fig6.update_layout(
    title  = '<b>Top 10 vs Bottom 10 Agents par Profit</b><br><span style="font-size:12px">Agent 127 : sinistres sans primes — anomalie à investiguer en priorité</span>',
    height = 620, width = 900, paper_bgcolor = 'white'
)
fig6.write_html('outputs/viz6_top_bottom_agents.html')
fig6.show()
print('✅ VIZ 6 — Top/Bottom agents sauvegardé.')

In [ ]:
# ── VIZ 7 : Distribution du Loss Ratio par contrat (Plotly histogram) ────────
lr_data = df_master[df_master['total_premium'] > 0]['loss_ratio']
# Filtre les contrats avec prime > 0 pour exclure les anomalies (agent 127).

n_deficitaire = (lr_data > 100).sum()
n_rentable    = (lr_data <= 100).sum()
# Comptage des contrats de chaque côté du seuil de rentabilité.

fig7 = go.Figure()
fig7.add_trace(go.Histogram(x=lr_data, nbinsx=60,
    marker_color='#2980B9', opacity=0.75, name='Contrats'))
# Histogramme de la distribution des Loss Ratios — révèle la forme du risque du portefeuille.

fig7.add_vrect(x0=0,   x1=80,  fillcolor='#27AE60', opacity=0.07, line_width=0)
fig7.add_vrect(x0=80,  x1=100, fillcolor='#E67E22', opacity=0.10, line_width=0)
fig7.add_vrect(x0=100, x1=lr_data.max(), fillcolor='#C0392B', opacity=0.07, line_width=0)
# Zones colorées : vert=sain, orange=limite, rouge=critique.

fig7.add_vline(x=100, line_dash='dash', line_color='#C0392B', line_width=2,
               annotation_text='Seuil 100%')
fig7.add_vline(x=lr_data.mean(), line_dash='dot', line_color='#E67E22',
               annotation_text=f"Moy: {lr_data.mean():.0f}%")
# Lignes de référence : seuil critique et moyenne du portefeuille.

fig7.add_annotation(
    x=0.97, y=0.92, xref='paper', yref='paper', align='right',
    text=f"Rentables (LR≤100%) : {n_rentable} ({n_rentable/len(lr_data)*100:.0f}%)<br>"
         f"Déficitaires (LR>100%) : {n_deficitaire} ({n_deficitaire/len(lr_data)*100:.0f}%)",
    showarrow=False, bgcolor='white', bordercolor='#95A5A6', borderwidth=1
)
fig7.update_layout(
    title  = '<b>Distribution du Loss Ratio par Contrat</b>',
    height = 460, width = 860, paper_bgcolor = 'white',
    xaxis  = dict(title='Loss Ratio (%)'),
    yaxis  = dict(title='Nombre de contrats')
)
fig7.write_html('outputs/viz7_distribution_lr.html')
fig7.show()
print('✅ VIZ 7 — Distribution LR sauvegardée.')

In [ ]:
# ── VIZ 8 : Treemap CA par Pays × Produit (Plotly) ───────────────────────────
tree = df_master.groupby(['country','product_type']).agg(
    CA   = ('total_premium',        'sum'),  # Superficie du rectangle = volume de primes.
    LR   = ('loss_ratio',           'mean'), # Couleur = intensité du Loss Ratio.
    Nb   = ('policy_id',            'count'),
    P    = ('profit',               'sum')
).reset_index()
tree['LR'] = tree['LR'].round(1)
# Agrégation double niveau : pays (parent) et produit (enfant) pour le treemap hiérarchique.

fig8 = px.treemap(
    tree,
    path   = [px.Constant('Portefeuille'), 'country', 'product_type'],
    # Hiérarchie : niveau global → pays → produit.
    values = 'CA',                 # Taille des blocs proportionnelle aux primes.
    color  = 'LR',                 # Couleur = Loss Ratio — rouge=destructeur, vert=sain.
    color_continuous_scale = [[0,'#27AE60'],[0.35,'#F39C12'],[0.65,'#E74C3C'],[1,'#7B241C']],
    color_continuous_midpoint = 100,
    hover_data = {'CA':':.0f', 'LR':':.1f', 'Nb':True, 'P':':.0f'},
    title  = '<b>Carte du Portefeuille : CA par Pays × Produit</b><br>Coloré par Loss Ratio — Rouge = destruction de valeur'
)
fig8.update_traces(
    textinfo = 'label+value',
    texttemplate = '%{label}<br>%{value:,.0f}',
    textfont = dict(size=11)
    # Affiche le nom et la valeur dans chaque bloc pour une lecture directe.
)
fig8.update_layout(
    height = 580, width = 920, paper_bgcolor = 'white',
    coloraxis_colorbar = dict(title='Loss Ratio %', ticksuffix='%')
)
fig8.write_html('outputs/viz8_treemap_portefeuille.html')
fig8.show()
print('✅ VIZ 8 — Treemap portefeuille sauvegardé.')

---
## 1️⃣2️⃣ Insights & Recommandations Stratégiques

### 🔴 Insights critiques

| # | Constat | Chiffre clé | Priorité |
|---|---|---|---|
| 1 | Loss Ratio global insoutenable | **189,3%** — norme : < 80% | 🔴 Critique |
| 2 | Habitation : quasi un sinistre par contrat | Fréquence **0,898** — LR **209%** | 🔴 Critique |
| 3 | Agent 127 : sinistres sans primes | Exposition **31 176** sans revenus | 🔴 Fraude potentielle |
| 4 | 2024 : pire année (LR 222%) + baisse CA -26% | Spirale déflationniste amorcée | 🔴 Urgent |
| 5 | Bottom 10% clients détruisent 2,65× la valeur du top 10% | -680K vs +257K | 🟠 Important |
| 6 | Email Marketing : ROI 231x avec 3,4% du budget | Levier sous-exploité | 🟢 Opportunité |
| 7 | Kenya : seul marché sous 160% de LR | -35 pts vs moyenne | 🟢 Développer |

---

### 🎯 5 Recommandations actionnables

**R1 — Auditer l'agent 127 et les bottom 5 agents Nigeria** *(0–30 jours · Urgent)*  
→ Exposition cumulée : 73 000 | **KPI :** Audit clôturé en 30 jours

**R2 — Geler Habitation et Vie au Maroc** *(0–60 jours · Urgent)*  
→ Stoppage immédiat des nouvelles souscriptions sur ces 2 segments | **KPI :** 0 nouvelle souscription

**R3 — Révision tarifaire globale — cible LR < 120%** *(1–3 mois)*  
→ Habitation ×2,09 | Vie ×2,07 | Auto ×1,78 | **KPI :** LR nouvelles souscriptions < 120% à M+6

**R4 — Renforcer les critères d'acceptation des sinistres** *(1–4 mois)*  
→ Taux actuel 80,9% → objectif < 65% | **KPI :** Taux acceptation < 68% à M+3

**R5 — Concentrer l'expansion sur Kenya × Santé** *(3–6 mois)*  
→ +200 contrats Kenya·Santé | **KPI :** LR global < 175% à M+12

In [ ]:
# ── Export df_master final ────────────────────────────────────────────────────
df_master.to_csv('outputs/df_master_final.csv', index=False, encoding='utf-8')
# Sauvegarde le dataset maître nettoyé et enrichi pour réutilisation future.

# ── Résumé final en console ───────────────────────────────────────────────────
print('=== FICHIERS GÉNÉRÉS ===')
for f in os.listdir('outputs'):
    size = os.path.getsize(f'outputs/{f}')
    print(f'  {f:<45} ({size/1024:.1f} KB)')
# Liste tous les livrables produits avec leur taille pour vérification.

print('\n✅ Analyse complète terminée.')
print(f'   Dataset maître : {df_master.shape[0]:,} lignes × {df_master.shape[1]} colonnes')
print(f'   Loss Ratio global : {lr_val:.1f}%')
print(f'   Prochaine étape : appliquer les 5 recommandations stratégiques.')